In [1]:
import hpelm as em
import numpy as np
import matplotlib.pyplot as plt
import pickle
from math import sqrt

from sklearn.metrics import mean_squared_error as MSE, mean_absolute_error as MAE, r2_score as R2, explained_variance_score as EVS
from sklearn.preprocessing import StandardScaler

%load_ext autoreload
%autoreload 2

%matplotlib inline

plt.rcParams['figure.figsize'] = (10, 8)

def fraction_within_eps(y_true, y_pred, epsilon=0.5):
    # Number of days where abs(y_true - y_pred) < epsilon divided by total number of days
    
    count = np.sum(np.abs(y_true - y_pred) <= epsilon)
    return count / y_true.shape[0]

yt = np.array([10, 10, 10, 10, 10])
yp = np.array([9.2, 9.7, 10.3, 10.2, 11])
print(fraction_within_eps(yt, yp, 0.6))
FIE = fraction_within_eps

def concorr(x, y):
    # Return Lin's concordance correlation coefficient
    # x, y are numpy arrays
    
    xm = x.mean()
    ym = y.mean()
    xv = x.var()
    yv = y.var()
    xycov = np.sum((x-xm)*(y-ym)) / x.shape[0]
    lin = 2*xycov / (xv + yv + (xm - ym)**2)
    return lin

xx = np.random.randn(10)
yy = xx + 10
print('R2 corr coef is {} whereas concordance corr coef is {}'.format(R2(yy,xx), concorr(yy,xx)))

0.6
R2 corr coef is -395.08581879628906 whereas concordance corr coef is 0.005024042318431473


In [13]:
with open('splitted_alldat.pkl', 'rb') as f:
    dictly = pickle.load(f)
    
x_train_ = dictly['x_train']
y_train = dictly['y_train']
x_val_ = dictly['x_dev']
y_val = dictly['y_dev']
x_test_ = dictly['x_test']
y_test = dictly['y_test']

# Convert to MJ
y_test *= (60/10**6)
y_val *= (60/10**6)
y_train *= (60/10**6)

In [3]:
# Scale with means and stds
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train_.astype(float))
x_val = scaler.transform(x_val_.astype(float))
x_test = scaler.transform(x_test_.astype(float))

print('Test Set | X Shape: {}, y shape: {}'.format(x_test_.shape, y_test.shape))
print('Validation Set | X Shape: {}, y shape: {}'.format(x_val_.shape, y_val.shape))
print('Training Set | X Shape: {}, y shape: {}'.format(x_train_.shape, y_train.shape))

# We don't need the validation set to be seperate here
x_train = np.concatenate([x_train, x_val])
x_train_ = np.concatenate([x_train_, x_val_])
y_train = np.concatenate([y_train, y_val])
print('Training Set Expanded | X Shape: {}, y shape: {}'.format(x_train.shape, y_train.shape))

Test Set | X Shape: (10601, 5), y shape: (10601,)
Validation Set | X Shape: (10797, 5), y shape: (10797,)
Training Set | X Shape: (58751, 5), y shape: (58751,)
Training Set Expanded | X Shape: (69548, 5), y shape: (69548,)


In [4]:
for neurons in range(5, 200, 5):
    model = em.ELM(5, 1)
    model.add_neurons(neurons, 'rbf_l2')
    model.train(x_train, y_train, 'LOO', 'OP')
    p_test = model.predict(x_test)
    #print('Test set squared error: {}'.format(model.error(y_test, p_test)))
    
    mse = MSE( y_test, p_test )
    rmse = sqrt( mse )
    mae = MAE( y_test, p_test )
    r2 = R2( y_test, p_test )
    evs = EVS( y_test, p_test )
    fie_h = FIE( y_test, np.squeeze(p_test), 0.5)
    fie_o = FIE( y_test, np.squeeze(p_test), 1)
    fie_oh = FIE( y_test, np.squeeze(p_test), 1.5)
    lin = concorr(y_test, np.squeeze(p_test))

    print('N: {:003d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
                  'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}, LINCC: {:.4f}'.format(neurons, rmse, mae, r2, evs,
                                                                                       fie_h, fie_o, fie_oh, lin))

N: 005 » RMSE: 6.8829, MAE: 5.2677, R2: 0.3723, EVS: 0.3752, FIE_H: 0.0528, FIE_O: 0.1118, FIE_OH: 0.1700, LINCC: 0.7397
N: 010 » RMSE: 5.4485, MAE: 3.9051, R2: 0.6067, EVS: 0.6373, FIE_H: 0.0969, FIE_O: 0.1891, FIE_OH: 0.2758, LINCC: 0.7841
N: 015 » RMSE: 5.0211, MAE: 3.3631, R2: 0.6659, EVS: 0.6822, FIE_H: 0.1135, FIE_O: 0.2224, FIE_OH: 0.3336, LINCC: 0.8170
N: 020 » RMSE: 5.2650, MAE: 3.5828, R2: 0.6327, EVS: 0.6457, FIE_H: 0.1126, FIE_O: 0.2175, FIE_OH: 0.3203, LINCC: 0.8063
N: 025 » RMSE: 4.5311, MAE: 2.7489, R2: 0.7280, EVS: 0.7500, FIE_H: 0.1707, FIE_O: 0.3313, FIE_OH: 0.4703, LINCC: 0.8499
N: 030 » RMSE: 4.5389, MAE: 2.5412, R2: 0.7270, EVS: 0.7515, FIE_H: 0.1900, FIE_O: 0.3641, FIE_OH: 0.5203, LINCC: 0.8539
N: 035 » RMSE: 4.8050, MAE: 3.0138, R2: 0.6941, EVS: 0.7120, FIE_H: 0.1389, FIE_O: 0.2738, FIE_OH: 0.3968, LINCC: 0.8372
N: 040 » RMSE: 4.4942, MAE: 2.4503, R2: 0.7324, EVS: 0.7592, FIE_H: 0.2268, FIE_O: 0.4222, FIE_OH: 0.5617, LINCC: 0.8564
N: 045 » RMSE: 4.6201, MAE: 2.55

In [14]:
# x column order: enlem - parlak saatler - ortalama sıcaklık - gün uzunluğu - h0 
frac_train = x_train_[:, 1] / x_train_[:, 3]
frac_val = x_val_[:, 1] / x_val_[:, 3]
frac_test = x_test_[:, 1] / x_test_[:, 3]

xtr = np.zeros((x_train_.shape[0], 2))
xtr[:, 0] = frac_train
xtr[:, 1] = x_train_[:, 4]

xvl = np.zeros((x_val_.shape[0], 2))
xvl[:, 0] = frac_val
xvl[:, 1] = x_val_[:, 4]

xte = np.zeros((x_test_.shape[0], 2))
xte[:, 0] = frac_test
xte[:, 1] = x_test_[:, 4]

# Scale with means and stds
scaler = StandardScaler()
x_train = scaler.fit_transform(xtr.astype(float))
x_val = scaler.transform(xvl.astype(float))
x_test = scaler.transform(xte.astype(float))

print('Test Set | X Shape: {}, y shape: {}'.format(x_test.shape, y_test.shape))
print('Validation Set | X Shape: {}, y shape: {}'.format(x_val.shape, y_val.shape))
print('Training Set | X Shape: {}, y shape: {}'.format(x_train.shape, y_train.shape))

# We don't need the validation set to be seperate here
x_train = np.concatenate([x_train, x_val])
y_train = np.concatenate([y_train, y_val])
print('Training Set Expanded | X Shape: {}, y shape: {}'.format(x_train.shape, y_train.shape))

Test Set | X Shape: (10601, 2), y shape: (10601,)
Validation Set | X Shape: (10797, 2), y shape: (10797,)
Training Set | X Shape: (58751, 2), y shape: (58751,)
Training Set Expanded | X Shape: (69548, 2), y shape: (69548,)


In [15]:
for neurons in range(5, 200, 5):
    model = em.ELM(2, 1)
    model.add_neurons(neurons, 'rbf_l2')
    model.train(x_train, y_train, 'LOO', 'OP')
    p_test = model.predict(x_test)
    #print('Test set squared error: {}'.format(model.error(y_test, p_test)))
    
    mse = MSE( y_test, p_test )
    rmse = sqrt( mse )
    mae = MAE( y_test, p_test )
    r2 = R2( y_test, p_test )
    evs = EVS( y_test, p_test )
    fie_h = FIE( y_test, np.squeeze(p_test), 0.5)
    fie_o = FIE( y_test, np.squeeze(p_test), 1)
    fie_oh = FIE( y_test, np.squeeze(p_test), 1.5)
    lin = concorr(y_test, np.squeeze(p_test))

    print('N: {:003d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
                  'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}, LINCC: {:.4f}'.format(neurons, rmse, mae, r2, evs,
                                                                                       fie_h, fie_o, fie_oh, lin))

N: 005 » RMSE: 7.0179, MAE: 5.3726, R2: 0.3474, EVS: 0.3522, FIE_H: 0.0651, FIE_O: 0.1284, FIE_OH: 0.1877, LINCC: 0.7329
N: 010 » RMSE: 4.9571, MAE: 3.0795, R2: 0.6744, EVS: 0.7065, FIE_H: 0.1411, FIE_O: 0.2841, FIE_OH: 0.4110, LINCC: 0.8265
N: 015 » RMSE: 4.7487, MAE: 3.0626, R2: 0.7012, EVS: 0.7228, FIE_H: 0.1273, FIE_O: 0.2609, FIE_OH: 0.3804, LINCC: 0.8355
N: 020 » RMSE: 4.4194, MAE: 2.3905, R2: 0.7412, EVS: 0.7707, FIE_H: 0.2243, FIE_O: 0.4188, FIE_OH: 0.5694, LINCC: 0.8619
N: 025 » RMSE: 4.4077, MAE: 2.3898, R2: 0.7426, EVS: 0.7723, FIE_H: 0.2184, FIE_O: 0.4190, FIE_OH: 0.5689, LINCC: 0.8623
N: 030 » RMSE: 4.4100, MAE: 2.3847, R2: 0.7423, EVS: 0.7721, FIE_H: 0.2188, FIE_O: 0.4232, FIE_OH: 0.5732, LINCC: 0.8622
N: 035 » RMSE: 4.4171, MAE: 2.3929, R2: 0.7415, EVS: 0.7715, FIE_H: 0.2146, FIE_O: 0.4238, FIE_OH: 0.5715, LINCC: 0.8619
N: 040 » RMSE: 4.4159, MAE: 2.3894, R2: 0.7416, EVS: 0.7716, FIE_H: 0.2198, FIE_O: 0.4228, FIE_OH: 0.5737, LINCC: 0.8621
N: 045 » RMSE: 4.4205, MAE: 2.39

In [3]:
# We need to pluck out the latitude and temperature columns, train that way for direct comparison with BSH quad model

# x column order: enlem - parlak saatler - ortalama sıcaklık - gün uzunluğu - h0 
x_train_ = x_train_[:, [1, 3, 4]]
x_val_ = x_val_[:, [1, 3, 4]]
x_test_ = x_test_[:, [1, 3, 4]]

# Scale with means and stds
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train_.astype(float))
x_val = scaler.transform(x_val_.astype(float))
x_test = scaler.transform(x_test_.astype(float))

print('Test Set | X Shape: {}, y shape: {}'.format(x_test_.shape, y_test.shape))
print('Validation Set | X Shape: {}, y shape: {}'.format(x_val_.shape, y_val.shape))
print('Training Set | X Shape: {}, y shape: {}'.format(x_train_.shape, y_train.shape))

# We don't need the validation set to be seperate here
x_train = np.concatenate([x_train, x_val])
y_train = np.concatenate([y_train, y_val])
print('Training Set Expanded | X Shape: {}, y shape: {}'.format(x_train.shape, y_train.shape))

Test Set | X Shape: (10601, 3), y shape: (10601,)
Validation Set | X Shape: (10797, 3), y shape: (10797,)
Training Set | X Shape: (58751, 3), y shape: (58751,)
Training Set Expanded | X Shape: (69548, 3), y shape: (69548,)


In [25]:
model = em.ELM(3, 1)
model.add_neurons(50, 'rbf_l2')
model.train(x_train, y_train, 'LOO', 'OP')
p_train = model.predict(x_train)
print(model.error(y_train, p_train))
p_test = model.predict(x_test)
print(model.error(y_test, p_test))

8.25883260885
19.4737406843


In [26]:
mse = MSE( y_test, p_test )
rmse = sqrt( mse )
mae = MAE( y_test, p_test )
r2 = R2( y_test, p_test )
evs = EVS( y_test, p_test )
fie_h = FIE( y_test, np.squeeze(p_test), 0.5)
fie_o = FIE( y_test, np.squeeze(p_test), 1)
fie_oh = FIE( y_test, np.squeeze(p_test), 1.5)
lin = concorr(y_test, np.squeeze(p_test))

print('RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
              'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}, LINCC: {:.4f}'.format(rmse, mae, r2, evs,
                                                                                   fie_h, fie_o, fie_oh, lin))

RMSE: 4.4129, MAE: 2.3833, R2: 0.7420, EVS: 0.7693, FIE_H: 0.2214, FIE_O: 0.4215, FIE_OH: 0.5690, LINCC: 0.8620


In [28]:
# Error decreases as number of neurons increase up to a point, then error increases again...

for neurons in range(5, 400, 5):
    model = em.ELM(3, 1)
    model.add_neurons(neurons, 'rbf_l2')
    model.train(x_train, y_train, 'LOO', 'OP')
    p_test = model.predict(x_test)
    #print('Test set squared error: {}'.format(model.error(y_test, p_test)))
    
    mse = MSE( y_test, p_test )
    rmse = sqrt( mse )
    mae = MAE( y_test, p_test )
    r2 = R2( y_test, p_test )
    evs = EVS( y_test, p_test )
    fie_h = FIE( y_test, np.squeeze(p_test), 0.5)
    fie_o = FIE( y_test, np.squeeze(p_test), 1)
    fie_oh = FIE( y_test, np.squeeze(p_test), 1.5)
    lin = concorr(y_test, np.squeeze(p_test))

    print('N: {:003d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
                  'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}, LINCC: {:.4f}'.format(neurons, rmse, mae, r2, evs,
                                                                                       fie_h, fie_o, fie_oh, lin))

N: 005 » RMSE: 7.3239, MAE: 5.3919, R2: 0.2893, EVS: 0.2893, FIE_H: 0.0765, FIE_O: 0.1489, FIE_OH: 0.2284, LINCC: 0.6118
N: 010 » RMSE: 5.9065, MAE: 4.2967, R2: 0.5378, EVS: 0.5672, FIE_H: 0.0840, FIE_O: 0.1688, FIE_OH: 0.2474, LINCC: 0.7365
N: 015 » RMSE: 5.4138, MAE: 3.5123, R2: 0.6117, EVS: 0.6431, FIE_H: 0.1107, FIE_O: 0.2238, FIE_OH: 0.3385, LINCC: 0.7959
N: 020 » RMSE: 4.4403, MAE: 2.4701, R2: 0.7388, EVS: 0.7675, FIE_H: 0.2084, FIE_O: 0.3936, FIE_OH: 0.5399, LINCC: 0.8594
N: 025 » RMSE: 4.4367, MAE: 2.4361, R2: 0.7392, EVS: 0.7663, FIE_H: 0.2072, FIE_O: 0.4004, FIE_OH: 0.5472, LINCC: 0.8609
N: 030 » RMSE: 4.4107, MAE: 2.4022, R2: 0.7422, EVS: 0.7708, FIE_H: 0.2173, FIE_O: 0.4163, FIE_OH: 0.5667, LINCC: 0.8619
N: 035 » RMSE: 4.3987, MAE: 2.3935, R2: 0.7436, EVS: 0.7712, FIE_H: 0.2192, FIE_O: 0.4159, FIE_OH: 0.5671, LINCC: 0.8632
N: 040 » RMSE: 4.4106, MAE: 2.4065, R2: 0.7422, EVS: 0.7707, FIE_H: 0.2155, FIE_O: 0.4147, FIE_OH: 0.5564, LINCC: 0.8621
N: 045 » RMSE: 4.4099, MAE: 2.39

/home/bulent/anaconda3/lib/python3.6/site-packages/hpelm/modules/mrsr.py:110: RuntimeWarning: invalid value encountered in true_divide
  G    = (cmax + S.dot(A)) / (cmax + S.dot(B))  # true slow for many outputs
/home/bulent/anaconda3/lib/python3.6/site-packages/hpelm/modules/mrsr.py:116: RuntimeWarning: invalid value encountered in greater_equal
  g = G[G>=0].min()


ValueError: zero-size array to reduction operation minimum which has no identity

In [29]:
# Error decreases as number of neurons increase up to a point, then error increases again...

for neurons in range(5, 400, 5):
    model = em.ELM(3, 1)
    model.add_neurons(neurons, 'rbf_l2')
    model.train(x_train, y_train, 'LOO', 'OP')
    p_test = model.predict(x_test)
    #print('Test set squared error: {}'.format(model.error(y_test, p_test)))
    
    mse = MSE( y_test, p_test )
    rmse = sqrt( mse )
    mae = MAE( y_test, p_test )
    r2 = R2( y_test, p_test )
    evs = EVS( y_test, p_test )
    fie_h = FIE( y_test, np.squeeze(p_test), 0.5)
    fie_o = FIE( y_test, np.squeeze(p_test), 1)
    fie_oh = FIE( y_test, np.squeeze(p_test), 1.5)
    lin = concorr(y_test, np.squeeze(p_test))

    print('N: {:003d} » RMSE: {:.4f}, MAE: {:.4f}, R2: {:.4f}, EVS: {:.4f}, '
                  'FIE_H: {:.4f}, FIE_O: {:.4f}, FIE_OH: {:.4f}, LINCC: {:.4f}'.format(neurons, rmse, mae, r2, evs,
                                                                                       fie_h, fie_o, fie_oh, lin))

N: 005 » RMSE: 5.3229, MAE: 3.7626, R2: 0.6246, EVS: 0.6384, FIE_H: 0.1026, FIE_O: 0.2054, FIE_OH: 0.2968, LINCC: 0.7974
N: 010 » RMSE: 5.0872, MAE: 3.2867, R2: 0.6571, EVS: 0.6798, FIE_H: 0.1177, FIE_O: 0.2298, FIE_OH: 0.3380, LINCC: 0.8192
N: 015 » RMSE: 4.7388, MAE: 2.7430, R2: 0.7025, EVS: 0.7384, FIE_H: 0.1748, FIE_O: 0.3317, FIE_OH: 0.4691, LINCC: 0.8403
N: 020 » RMSE: 4.5966, MAE: 2.7579, R2: 0.7200, EVS: 0.7455, FIE_H: 0.1653, FIE_O: 0.3286, FIE_OH: 0.4591, LINCC: 0.8494
N: 025 » RMSE: 4.5029, MAE: 2.5525, R2: 0.7313, EVS: 0.7585, FIE_H: 0.1931, FIE_O: 0.3705, FIE_OH: 0.5138, LINCC: 0.8563
N: 030 » RMSE: 4.3855, MAE: 2.3872, R2: 0.7452, EVS: 0.7724, FIE_H: 0.2139, FIE_O: 0.4146, FIE_OH: 0.5641, LINCC: 0.8634
N: 035 » RMSE: 4.3910, MAE: 2.3933, R2: 0.7445, EVS: 0.7735, FIE_H: 0.2213, FIE_O: 0.4209, FIE_OH: 0.5615, LINCC: 0.8635
N: 040 » RMSE: 4.4265, MAE: 2.4019, R2: 0.7404, EVS: 0.7685, FIE_H: 0.2236, FIE_O: 0.4182, FIE_OH: 0.5608, LINCC: 0.8612
N: 045 » RMSE: 4.4167, MAE: 2.40

/home/bulent/anaconda3/lib/python3.6/site-packages/hpelm/modules/mrsr.py:110: RuntimeWarning: invalid value encountered in true_divide
  G    = (cmax + S.dot(A)) / (cmax + S.dot(B))  # true slow for many outputs
/home/bulent/anaconda3/lib/python3.6/site-packages/hpelm/modules/mrsr.py:116: RuntimeWarning: invalid value encountered in greater_equal
  g = G[G>=0].min()


ValueError: zero-size array to reduction operation minimum which has no identity